# Chapter 5: OLS Asymptotics
### Heiss & Brunner (2024)

## What this chapter covers
1. WHY asymptotics — what happens when normality fails?
2. Consistency of OLS — verified with simulation
3. Asymptotic normality — the CLT at work
4. RESET test for functional form
5. LM test

**Format:** One step per cell. Manual calculation first, then statsmodels, then comparison.

## Cell 1 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
import statsmodels.stats.diagnostic as diag
import wooldridge as woo

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', os.getcwd())
print('Output directory :', output_dir)

## Cell 2 — Load wage1, explore it

In [ ]:
df = woo.dataWoo('wage1')
df['lwage'] = np.log(df['wage'])
print('Shape:', df.shape)

In [ ]:
print('First 8 rows:')
print(df[['wage','educ','exper','tenure','female']].head(8).to_string())

In [ ]:
print('Summary statistics:')
print(df[['wage','educ','exper','tenure']].describe().round(3).to_string())

---
## Section 1 — WHY Asymptotics?

In Chapters 3–4 we assumed MLR.6: errors are exactly normal. In practice:
- Wages are right-skewed
- Binary outcomes are 0/1
- Counts are non-negative integers

**The solution:** Large-sample theory. Under weaker assumptions, OLS is:
1. **Consistent** — converges to truth as n → ∞
2. **Asymptotically normal** — approximately normal in large samples (CLT)

This justifies t and F tests without normality, as long as n is large enough.

---
## Section 2 — Consistency of OLS

### WHY consistency?
Consistency means β_hat_j → β_j as n → ∞.
Requires only Cov(x_j, u) = 0 — weaker than full zero conditional mean.

**Key:** Inconsistency (from OVB, measurement error) does NOT go away with more data.

### Example Question
> Simulate y = 1 + 0.5x + u where u ~ chi²(3)−3 (NON-NORMAL).
> Show that OLS is still consistent: estimates converge to the true values.

### Step 2.1 — Set the true DGP

In [ ]:
np.random.seed(42)
true_b0 = 1.0
true_b1 = 0.5
print('True DGP: y = 1 + 0.5*x + u')
print('u ~ chi2(3) - 3  (mean=0, non-normal)')
print('True beta0 =', true_b0)
print('True beta1 =', true_b1)

### Step 2.2 — Simulate for different sample sizes

In [ ]:
sample_sizes = [20, 50, 100, 500, 1000, 5000]
n_sims = 2000
results_by_n = {}
for n in sample_sizes:
    b1_vals = []
    for _ in range(n_sims):
        x = np.random.uniform(0, 10, n)
        u = np.random.chisquare(3, n) - 3
        y = true_b0 + true_b1*x + u
        xd = x - x.mean()
        b1 = np.sum(xd*(y-y.mean()))/np.sum(xd**2)
        b1_vals.append(b1)
    results_by_n[n] = np.array(b1_vals)
print('Simulation complete.')

### Step 2.3 — Show the results table

In [ ]:
print(f'{"n":>6} | {"Mean(b1_hat)":>14} | {"Std(b1_hat)":>12} | {"Bias":>10}')
print('-'*50)
for n in sample_sizes:
    arr = results_by_n[n]
    print(f'{n:>6} | {arr.mean():>14.6f} | {arr.std():>12.6f} | {arr.mean()-true_b1:>10.6f}')
print(f'\nTrue beta1 = {true_b1}')
print('As n increases: mean approaches 0.5 and bias shrinks to 0 -> CONSISTENT')

### Step 2.4 — Plot the convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, 4))
for n, col in zip([20, 100, 1000, 5000], colors):
    axes[0].hist(results_by_n[n], bins=60, density=True, alpha=0.55, color=col, label=f'n={n}')
axes[0].axvline(true_b1, color='red', lw=2.5, linestyle='--', label=f'True={true_b1}')
axes[0].set_title('OLS Consistency: Distribution Collapses on Truth', fontsize=11, fontweight='bold')
axes[0].set_xlabel('beta1_hat')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8)
stds = [results_by_n[n].std() for n in sample_sizes]
axes[1].loglog(sample_sizes, stds, 'bo-', lw=2, markersize=8, label='Std of beta1_hat')
ref = np.array(sample_sizes)
axes[1].loglog(ref, stds[0]*np.sqrt(sample_sizes[0]/ref), 'r--', lw=2, label='1/sqrt(n) reference')
axes[1].set_title('Std Dev Falls as 1/sqrt(n)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Sample size (log)')
axes[1].set_ylabel('Std Dev (log)')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch05_01_consistency.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Asymptotic Normality

### WHY it matters
Even without normal errors, for large n:
$$\sqrt{n}(\hat{\beta}_1 - \beta_1) \xrightarrow{d} N(0, \sigma^2/\text{Var}(x))$$

This justifies t and F tests even with skewed/non-normal residuals.

### Example Question
> Simulate from the chi-squared DGP. Show that the t-statistic distribution approaches N(0,1) as n grows.

### Step 3.1 — Run simulation for different n

In [ ]:
np.random.seed(42)
n_sims_an = 4000
sizes_an = [30, 100, 500]
t_stats_by_n = {}
for n in sizes_an:
    t_vals = []
    for _ in range(n_sims_an):
        x = np.random.uniform(0, 10, n)
        u = np.random.chisquare(3, n) - 3
        y = true_b0 + true_b1*x + u
        df_s = pd.DataFrame({'y':y,'x':x})
        res = smf.ols('y ~ x', data=df_s).fit()
        t_vals.append(res.tvalues['x'])
    t_stats_by_n[n] = np.array(t_vals)
print('Simulation complete.')

### Step 3.2 — Plot: t-stats approach N(0,1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, n in zip(axes, sizes_an):
    t_arr = t_stats_by_n[n]
    ax.hist(t_arr, bins=60, density=True, color='steelblue', edgecolor='white', alpha=0.8)
    x_range = np.linspace(-5, 5, 200)
    ax.plot(x_range, stats.norm.pdf(x_range), 'r-', lw=2.5, label='N(0,1)')
    ax.plot(x_range, stats.t.pdf(x_range, df=n-2), 'g--', lw=2, label=f't({n-2})')
    ax.set_title(f'n={n} — non-normal errors', fontsize=11, fontweight='bold')
    ax.set_xlabel('t-statistic')
    if n == 30: ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.set_xlim(-5, 5)
plt.suptitle('Asymptotic Normality: t-stats approach N(0,1) even with chi-squared errors', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch05_02_asymp_normality.png'), dpi=150, bbox_inches='tight')
plt.show()
print('For n=30: somewhat non-normal (fat tails)')
print('For n=100+: very close to N(0,1)')
print('-> t and F tests are valid in large samples even without normality')

---
## Section 4 — RESET Test for Functional Form

### WHY the RESET test?
If the true relationship is nonlinear but we fit a linear model, OLS is misspecified.

### Procedure (4 steps)
1. Estimate original model → get ŷ
2. Add ŷ² and ŷ³ to the regression
3. F-test: H₀: coefficients on ŷ² and ŷ³ are jointly zero
4. Rejection → functional form is wrong

### Example Question
> Apply RESET to `wage ~ educ` (level-level). Does it pass?
> Then apply to `lwage ~ educ` (log-level). Which model is better specified?

### Step 4.1 — Run the original level-level model

In [ ]:
res_ll = smf.ols('wage ~ educ', data=df).fit()
print('Level-level model: wage ~ educ')
print(f'  beta0 = {res_ll.params["Intercept"]:.4f}')
print(f'  beta1 = {res_ll.params["educ"]:.4f}')
print(f'  R2    = {res_ll.rsquared:.4f}')

### Step 4.2 — Manually add ŷ² and ŷ³

In [ ]:
y_hat_ll = res_ll.fittedvalues
df_reset = df.copy()
df_reset['yhat2'] = y_hat_ll ** 2
df_reset['yhat3'] = y_hat_ll ** 3
print('Fitted values (first 5):', y_hat_ll.values[:5].round(4))
print('yhat^2  (first 5):', df_reset['yhat2'].values[:5].round(4))
print('yhat^3  (first 5):', df_reset['yhat3'].values[:5].round(4))

### Step 4.3 — Run augmented regression

In [ ]:
res_aug = smf.ols('wage ~ educ + yhat2 + yhat3', data=df_reset).fit()
print('Augmented regression (with yhat^2 and yhat^3):')
print(res_aug.summary())

### Step 4.4 — Manual F-test: are yhat² and yhat³ jointly zero?

In [ ]:
SSR_r  = res_ll.ssr
SSR_ur = res_aug.ssr
q      = 2
n_obs  = int(res_aug.nobs)
k_ur   = len(res_aug.params) - 1
df_den = n_obs - k_ur - 1
F_manual = ((SSR_r - SSR_ur)/q) / (SSR_ur/df_den)
p_manual = stats.f.sf(F_manual, dfn=q, dfd=df_den)
print('RESET F-statistic (manual):')
print(f'  SSR_restricted  = {SSR_r:.4f}')
print(f'  SSR_unrestricted= {SSR_ur:.4f}')
print(f'  q={q}, df_den={df_den}')
print(f'  F = {F_manual:.4f}')
print(f'  p = {p_manual:.6f}')

### Step 4.5 — Compare with statsmodels f_test

In [ ]:
f_sm = res_aug.f_test('yhat2=0, yhat3=0')
F_sm = f_sm.fvalue[0][0]
p_sm = f_sm.pvalue
print('F-statistic (statsmodels):', round(F_sm, 4))
print('p-value     (statsmodels):', round(p_sm, 6))
print()
print('Manual vs statsmodels:')
print(f'  F manual={F_manual:.4f}  F sm={F_sm:.4f}  match={np.isclose(F_manual, F_sm)}')
print(f'  p manual={p_manual:.6f}  p sm={p_sm:.6f}  match={np.isclose(p_manual, p_sm)}')
print()
decision = 'REJECT H0 -> functional form is MISSPECIFIED' if p_manual < 0.05 else 'FAIL TO REJECT H0'
print(f'Decision (5%): {decision}')

### Step 4.6 — Now test the log-level model

In [ ]:
res_log = smf.ols('lwage ~ educ', data=df).fit()
y_hat_log = res_log.fittedvalues
df_reset2 = df.copy()
df_reset2['yhat2_log'] = y_hat_log ** 2
df_reset2['yhat3_log'] = y_hat_log ** 3
res_aug2 = smf.ols('lwage ~ educ + yhat2_log + yhat3_log', data=df_reset2).fit()
SSR_r2   = res_log.ssr
SSR_ur2  = res_aug2.ssr
F_log    = ((SSR_r2-SSR_ur2)/2) / (SSR_ur2/(n_obs-3-1))
p_log    = stats.f.sf(F_log, dfn=2, dfd=n_obs-4)
print('RESET for log-level model:')
print(f'  F = {F_log:.4f}')
print(f'  p = {p_log:.6f}')
decision2 = 'REJECT H0 -> misspecified' if p_log < 0.05 else 'FAIL TO REJECT H0 -> functional form OK'
print(f'  Decision (5%): {decision2}')

### Step 4.7 — Summary: level-level vs log-level

In [ ]:
print('RESET TEST SUMMARY:')
print(f'  Level-level (wage~educ)  : F={F_manual:.4f}, p={p_manual:.4f} -> FAILS RESET (nonlinearity detected)')
print(f'  Log-level (lwage~educ)   : F={F_log:.4f}, p={p_log:.4f} -> {"FAILS" if p_log<0.05 else "PASSES"} RESET')
print()
print('CONCLUSION: Use log(wage) as the dependent variable.')
print('The level-level model has a functional form problem.')
result_df = pd.DataFrame({'Model':['wage~educ','lwage~educ'],'F':np.round([F_manual,F_log],4),'p':np.round([p_manual,p_log],4),'Decision':['Fails RESET','Passes RESET' if p_log>=0.05 else 'Fails RESET']})
result_df.to_csv(os.path.join(output_dir, 'table_ch05_RESET.csv'), index=False)
print('\n-> Saved to output/table_ch05_RESET.csv')

---
## Chapter 5 Summary

| Property | Requires | Key result |
|----------|----------|------------|
| Unbiasedness | E(u|x)=0 | Exact, any n |
| Consistency | Cov(x,u)=0 | Large sample |
| Asymptotic normality | + finite variance | t/F valid for large n |
| RESET | — | Detects nonlinearity |

### Practice Questions
1. Simulate with logistic errors (fat tails). Are t-tests still valid at n=100?
2. Run RESET for `lwage ~ educ + exper + tenure`. Does the full model pass?
3. Why does consistency NOT require normality of u?
4. At what sample size does the t-statistic distribution look approximately normal?